In [7]:
import io

import mysql.connector
import pandas as pd
from mysql.connector import MySQLConnection

In [8]:
credentials = {
  "user": "external.data.user",
  "password": "-----------------",
  "host": "199.192.16.220",
  "port": 3307
}

conn: MySQLConnection = mysql.connector.connect(**credentials)

In [9]:
def write_df_to_db(conn: MySQLConnection, df: pd.DataFrame, guid: str):
  buf = io.BytesIO()
  df.to_feather(buf)
  buf.seek(0)
  b = buf.read()

  cur = conn.cursor()
  cur.execute("""
  DELETE FROM external_data.dms WHERE guid = %s
  """, [guid])
  cur.execute("""
  INSERT INTO external_data.dms (guid, data) VALUES (%s, %s)
  """, [guid, b])
  cur.close()
  conn.commit()

In [10]:
df = pd.DataFrame(data={'id': [1, 2], 'val1': ['a', 'b'], 'val2': ['c', 'd']})
df_guid = '8c3a480d-6741-4e68-9319-7eb3a956ec21'

In [11]:
write_df_to_db(conn, df, df_guid)

In [12]:
conn.close()